In [48]:
import os
from bs4 import BeautifulSoup as Soup, Tag
import re

In [50]:
def collect_paths(dirname='.', extension='') -> list:
    """
    This is a function that collects paths and
    filenames across all folders in a parent folder
    """
    paths, filenames = [], []
    for address, _, files in os.walk(dirname):
        for filename in files:
            # Обработка без фильтра расширения
            if extension == '':
                filenames.append(filename)
                paths.append(os.path.join(address, filename))
            # Обработка с фильтром расширения
            else:
                if extension in filename:
                    filenames.append(filename)
                    paths.append(os.path.join(address, filename))
    return paths, filenames

def compile_chapter(section: Tag):
    title = title_tag.get_text(strip=True) if (
        title_tag := section.find('title')
    ) else ''
    rows = '\n'.join(
        [p.get_text(strip=True) for p in title_tag.find_next_siblings('p')]
    )
    return f'{title}\n{rows}'

def is_russian_letter(c):
    return ('А' <= c <= 'я') or c in 'Ёё'

In [51]:
paths, filenames = collect_paths('books', 'fb2')
book_counter = 0
deleted_characters_counter = 0
problem_paragraphs = []
books_skipped = 0

# Clear old file
if os.path.exists("all_texts.txt"):
  os.remove("all_texts.txt")

for path, filename in zip(paths, filenames):

    print(f"Parsing {filename}")

    try:
        # Get all valid paragraphs
        with open(path, 'rb') as f:
            soup = Soup(f, 'lxml-xml')
        body = soup.find('body')
        paragraphs = [
            p.get_text(strip=True)
            for p in body.find_all('p')
            if p.parent.name not in ('title', 'epigraph')
        ]

        # Delete non-russian letters
        cleaned_paragraphs = []
        for paragraph in paragraphs:
            words = re.sub(r'[«»_…]', '', paragraph).split()
            cleaned_words = [word for word in words if (not word[0].isalpha()) or (word[0].isalpha() and is_russian_letter(word[0]))]
            deleted_characters_counter += len(words) - len(cleaned_words)
            if ' '.join(cleaned_words) and  ' '.join(cleaned_words) != '—':
                cleaned_paragraphs.append(' '.join(cleaned_words))
            if len(cleaned_words) != len(words):
                problem_paragraphs.append(' '.join(words))

        # Write results to a common file
        with open("all_texts.txt", "a", encoding='utf-8') as f:
            for paragraph in cleaned_paragraphs:
                f.write(f"{paragraph}\n")

        # Write results to a specific file
        root, ext = os.path.splitext(filename)
        with open(os.path.join('outputs', root + '.txt'), "w", encoding='utf-8') as f:
            for paragraph in cleaned_paragraphs:
                f.write(f"{paragraph}\n")

        book_counter += 1

    except:
        books_skipped += 1

# log results
print(f"Process successfully completed!")
print(f"Books parsed:  {book_counter}")
print(f"Books skipped: {books_skipped}")
print(f"Words deleted: {deleted_characters_counter}")

# log problems
with open("problems.txt", "w", encoding='utf-8') as f:
        for paragraph in problem_paragraphs:
            f.write(f"{paragraph}\n")

Parsing Том 1. Повести и рассказы 1846-1847.fb2
Parsing Том 10. Братья Карамазовы. Том 2. Неоконченное. Стихотворения.fb2
Parsing Том 11. Публицистика 1860-х годов.fb2
Parsing Том 12. Дневник писателя 1873. Статьи и очерки.fb2
Parsing Том 13. Дневник писателя. 1876.fb2
Parsing Том 14. Дневник писателя 1877, 1980, 1981.fb2
Parsing Том 15. Письма 1834-1881.fb2
Parsing Том 2. Повести и рассказы 1848-1852.fb2
Parsing Том 3. Село Степанчиково и его обитатели. Записки из Мертвого дома.fb2
Parsing Том 4. Униженные и оскорбленные. Повести и рассказы 1862–1866. Игрок.fb2
Parsing Том 5. Преступление и наказание.fb2
Parsing Том 6. Идиот.fb2
Parsing Том 7. Бесы.fb2
Parsing Том 8. Вечный муж. Подросток.fb2
Parsing Том 9. Братья Карамазовы. Том 1.fb2
Parsing Marina_Tsvetaeva_Poyemyu.fb2
Parsing Marina_Tsvetaeva_Poyemyu_skazki.fb2
Parsing Marina_Tsvetaeva_Poyeticheskie_perevodyu.fb2
Parsing Marina_Tsvetaeva_Stihotvoreniya_1901_1916.fb2
Parsing Marina_Tsvetaeva_Stihotvoreniya_1916_1920.fb2
Parsing Mar

In [46]:
lst = '«Отцов , — Поди, поди прочь, anime — прошептала она, — ты пьяный и злой! Ты не гость мне!.. — Тут она снова обратилась к старику и опять приковалась к нему своими очами.'
print(bool(' '.join(['', ''])))

True
